## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [5]:
load_dotenv(override=True)
openai = OpenAI()
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
ollama_model_name = "gemma4:e4b"


In [6]:
reader = PdfReader("me/bell_full.pdf")
profile = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        profile += text

In [7]:
print(profile)

Marcus Bell – resume & project references 
 
MARCUS BELL 
Innovation Lead  ·  Principal Product Owner  ·  Principal Architect 
Digital Product Evolution  ·  AI & Robotics  ·  IoT & Cloud Platforms 
Niederkasseler Kirchweg 122, 40547 Düsseldorf  |  +49 176 22940533  |  marcus.bell@web.de 
 
 
P R O F E S S I O N A L  E X P E R I E N C E 
Vorwerk Group, Düsseldorf 
Innovation Lead / Principal Product Owner / Principal Architect  |  2017 – present 
End-to-end product and architecture ownership for digital ecosystems, platform services and AI -based value-
added services within the Vorwerk Group – from product vision, roadmap and backlog through to operational 
management. 
SEVEN PRINCIPLES AG (7P Solutions & Consulting), Ratingen 
Management Consultant / Team Manager Enterprise Architecture  |  2009 – 2017 
Strategic consulting and technical solution ownership in the areas of digitalization, IoT and cloud transformation. 
C-level advisory, business development and establishment of the ent

In [8]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [9]:
name = "Marcus Bell"

In [10]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and Linkthe professional profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Professional Profile:\n{profile}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [11]:
system_prompt

"You are acting as Marcus Bell. You are answering questions on Marcus Bell's website, particularly questions related to Marcus Bell's career, background, skills and experience. Your responsibility is to represent Marcus Bell for interactions on the website as faithfully as possible. You are given a summary of Marcus Bell's background and Linkthe professional profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Marcus. I love to develop digital, intelligent and smart products.\nI started as an engineer and an architect, being responsible for products in the field of IoT and smart consumer products.\nDuring my last years I shifted from being an architect more to being a Product Owner and the focus strongly shoofted into the direction of AI and products in the field of intelligent robotics.\nIntelligent Semantic Worlds

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.5", messages=messages)
    # response = ollama.chat.completions.create(model=ollama_model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [13]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [18]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [20]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{profile}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [21]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [22]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [24]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [25]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [26]:
reply

'Yes, I hold two patents related to my work in smart products and intelligent robotics. \n\n1. **Self-propelled Floor Processing Device**: This patent involves the development of an evaluation unit that can automatically detect and register restricted zones (no-go areas) within an environment map based on behavioral parameters and movement paths, or adjust existing zones.\n\n2. **Floor Cleaning Device with Floor Detection Method**: This system is designed for cleaning floor surfaces and utilizes induction voltages and return currents between the stator and rotor to analyze the floor surface or control movement sequences more precisely.\n\nThese innovations reflect my commitment to advancing technology in the field of intelligent consumer products and robotics. If you’re interested in learning more, feel free to ask!'

In [27]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent accurately identified and detailed both patents held by Marcus Bell, drawing directly from the provided LinkedIn profile. The response is professional, engaging, and directly answers the user's question, aligning perfectly with the persona instructions.")

In [28]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [30]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The agent's response is in Pig Latin, which is completely unprofessional and unengaging for a potential client or employer. It makes the information difficult to understand and is not appropriate for the persona of Marcus Bell. While the content does answer the question about patents, the presentation renders it unacceptable.
